# Prerequisite
Service Principal as Log Analytics Data Reader and Monitoring Reader

In [1]:
# DROP TABLE dbo.tokenConsumption;
# CREATE TABLE IF NOT EXISTS dbo.tokenConsumption
# (
#     last_TimeGenerated STRING,
#     timeday DATE,
#     foundryAccount STRING,
#     foundryProject STRING,
#     gen_ai_agent_id STRING,
#     user_mail STRING,
#     trace_conversation_id STRING,
#     cached_tokens BIGINT,
#     input_tokens BIGINT,
#     output_tokens BIGINT,
#     gen_ai_response_model STRING,
#     microsoft_foundry_project_id STRING
# )

StatementMeta(, 850c7599-f201-486f-a891-41db346fc92e, 3, Finished, Available, Finished, False)

In [2]:
%pip install azure-monitor-query azure-identity

from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient
import pandas as pd
from datetime import timedelta

StatementMeta(, 850c7599-f201-486f-a891-41db346fc92e, 9, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 3.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [3]:
import time
from azure.core.credentials import AccessToken, TokenCredential
from azure.keyvault.secrets import SecretClient

## SP for authentication to the Foundry Project
tenant_id = "94e2fbfd-171c-4f8c-9b70-fbe8002dac58"
client_id = "e6158f09-b7d6-426f-8613-711fd3d2c6f2"

# --- Key Vault (behind the private endpoint) ---
key_vault_url = "https://gelatai-in-keyvault.vault.azure.net/"  # the vault your connection pointed at
secret_name   = "sp-fabriceval-secret"

# Reuse the Fabric executing identity (workspace identity in pipeline runs / your user interactively)
class FabricTokenCredential(TokenCredential):
    def get_token(self, *scopes, **kwargs):
        token = notebookutils.credentials.getToken("keyvault")   # AAD token for the KV audience
        return AccessToken(token, int(time.time()) + 30 * 60)    # let the SDK refresh well before 1h expiry

secret_client = SecretClient(vault_url=key_vault_url, credential=FabricTokenCredential())
client_secret = secret_client.get_secret(secret_name).value

StatementMeta(, 850c7599-f201-486f-a891-41db346fc92e, 11, Finished, Available, Finished, False)

In [4]:
# -----------------------------------------
# CONFIGURATION
# -----------------------------------------

WORKSPACE_ID = "d0daaded-0ab0-44ea-957d-c8dec22363ea"

DEST_TABLE = "dbo.tokenConsumption"

LAW_QUERY_TEMPLATE = """
// Trace token used by conversation_id and user
AppDependencies
| where TimeGenerated > datetime("{watermark}")
| where Data in ("chat")
| extend dependencies_conversation_id = tostring(Properties.["gen_ai.conversation.id"])
// | where dependencies_conversation_id == "conv_0b9cf8dea7a0ddec00HS74kJ0FMdXQt5RTpRxMe3fDXHaaVY8s"
| extend dependencies_response_id = tostring(Properties.["gen_ai.response.id"])
| join kind=leftouter (
    AppTraces
    | where TimeGenerated > datetime("{watermark}")
    | where Message == "gen_ai.tracing"
    | extend user_id = tostring(Properties.enduser_id), user_mail = tostring(Properties.enduser_mail), trace_conversation_id = tostring(Properties.["foundry.conversation_id"])
    | project user_id, user_mail, trace_conversation_id
) on $left.dependencies_conversation_id == $right.trace_conversation_id
| project TimeGenerated, Measurements, user_id, user_mail, trace_conversation_id, Properties
| extend cached_tokens = toint(Measurements.["gen_ai.usage.cached_tokens"]), input_tokens = toint(Measurements.["gen_ai.usage.input_tokens"]), output_tokens = toint(Measurements.["gen_ai.usage.output_tokens"]),
gen_ai_agent_id = tostring(Properties.["gen_ai.agent.id"]), gen_ai_response_model = tostring(Properties.["gen_ai.response.model"]),
microsoft_foundry_project_id = tostring(Properties.["microsoft.foundry.project.id"])
, timeday = startofday(TimeGenerated)
| summarize last_TimeGenerated = tostring(max(TimeGenerated)), cached_tokens = sum(cached_tokens), input_tokens = sum(input_tokens), output_tokens = sum(output_tokens) by user_id, user_mail, trace_conversation_id, gen_ai_agent_id, gen_ai_response_model, microsoft_foundry_project_id, timeday
| extend foundryAccount = extract(@"/accounts/([^/]+)", 1, microsoft_foundry_project_id), foundryProject = extract(@"/projects/([^/]+)", 1, microsoft_foundry_project_id)
| project last_TimeGenerated, timeday, foundryAccount, foundryProject, gen_ai_agent_id, user_mail, trace_conversation_id, cached_tokens, input_tokens, output_tokens, gen_ai_response_model, microsoft_foundry_project_id
"""

# -----------------------------------------
# READ WATERMARK FROM LAKEHOUSE
# -----------------------------------------

from pyspark.sql import functions as F

try:
    watermark_df = spark.sql(f"SELECT MAX(last_TimeGenerated) as watermark FROM dbo.tokenConsumption")

    watermark = watermark_df.collect()[0]["watermark"]

except:

    watermark = None

# first execution
if watermark is None:
    watermark = "2026-07-01T00:00:00Z"
# else:
#     watermark = watermark.strftime("%Y-%m-%dT%H:%M:%S:%ssZ")

print(f"Watermark: {watermark}")

StatementMeta(, 850c7599-f201-486f-a891-41db346fc92e, 12, Finished, Available, Finished, False)

Watermark: 2026-07-08T09:29:28.9557956Z


In [5]:
# -----------------------------------------
# QUERY LOG ANALYTICS
# -----------------------------------------

query = LAW_QUERY_TEMPLATE.format(
    watermark=watermark
)

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

client = LogsQueryClient(credential)

response = client.query_workspace(
    workspace_id=WORKSPACE_ID,
    query=query,
    timespan=timedelta(days=90)
)

StatementMeta(, 850c7599-f201-486f-a891-41db346fc92e, 13, Finished, Available, Finished, False)

In [6]:
# no new rows
if len(response.tables) == 0:
    print("No rows returned")
else:

    table = response.tables[0]

    pdf = pd.DataFrame(
        table.rows,
        columns=table.columns
    )

    print(f"Retrieved {len(pdf)} rows")

# -----------------------------------------
# WRITE TO LAKEHOUSE (APPEND)
# -----------------------------------------

if len(pdf) == 0:
    print("No rows to add")

else:
    sdf = spark.createDataFrame(pdf)

    sdf = sdf.withColumn(
        "timeday",
        F.to_date("timeday")
    )

    (
        sdf.write
        .mode("append")
        .saveAsTable(DEST_TABLE)
    )

    print("Append completed")

StatementMeta(, 850c7599-f201-486f-a891-41db346fc92e, 14, Finished, Available, Finished, False)

Retrieved 0 rows
No rows to add
